# Baseline Models — Telco Churn
DummyClassifier e LogisticRegression com StratifiedKFold + MLflow tracking.

In [2]:
import logging
import warnings
warnings.filterwarnings("ignore")

import mlflow
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, classification_report,
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

SEED = 42
np.random.seed(SEED)

DATA_PATH = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
TARGET = "Churn Value"

NUMERIC_FEATURES = ["Tenure Months", "Monthly Charges", "Total Charges"]
CATEGORICAL_FEATURES = [
    "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method",
]

## 1. Carregar e preparar os dados

In [3]:
df = pd.read_csv(DATA_PATH)

# Colunas de cobrança chegam como string no CSV — forçar conversão para float
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df["Monthly Charges"] = pd.to_numeric(df["Monthly Charges"], errors="coerce")
df["Tenure Months"] = pd.to_numeric(df["Tenure Months"], errors="coerce")

X = df[CATEGORICAL_FEATURES + NUMERIC_FEATURES]
y = df[TARGET].values

print(f"Contagem de classes: {np.bincount(y)}")
print(f"Shape: {X.shape} | Churn rate: {y.mean():.2%}")
X.head()

Contagem de classes: [5174 1869]
Shape: (7043, 19) | Churn rate: 26.54%


,Gender,Senior Citizen,Partner,Dependents,Phone Service,Multiple Lines,Internet Service,Online Security,Online Backup,Device Protection,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Tenure Months,Monthly Charges,Total Charges
0,Male,No,No,No,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,2,53.85,108.15
1,Female,No,No,Yes,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,2,70.70,151.65
2,Female,No,No,Yes,Yes,Yes,Fiber optic,No,No,Yes,No,Yes,Yes,Month-to-month,Yes,Electronic check,8,99.65,820.50
3,Female,No,Yes,Yes,Yes,Yes,Fiber optic,No,No,Yes,Yes,Yes,Yes,Month-to-month,Yes,Electronic check,28,104.80,3046.05
4,Male,No,No,Yes,Yes,Yes,Fiber optic,No,Yes,Yes,No,Yes,Yes,Month-to-month,Yes,Bank transfer (automatic),49,103.70,5036.30


## 2. Preprocessamento

In [4]:
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")), # Preenche NaNs com a mediana — robusta a outliers
    ("scaler", StandardScaler()),                  # Centraliza (média=0) e normaliza (desvio=1) para não distorcer o modelo (Sigmoid é sensível a escala)
])
categorical_pipe = Pipeline([
    ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)), # Transforma categorias em colunas binárias; ignora categorias novas no momento da inferência
])
# Aplica cada pipeline apenas nas colunas correspondentes, mantendo o restante intacto
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, NUMERIC_FEATURES),
        ("cat", categorical_pipe, CATEGORICAL_FEATURES),
    ]
)

## 3. Treinar e avaliar modelos

In [5]:
# stratify=y garante que a proporção de churn (~26%) seja mantida em treino e teste
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=SEED)

# --- DummyClassifier: baseline ingênuo que sempre prevê a classe majoritária ---
dummy = Pipeline([("pre", preprocessor), ("clf", DummyClassifier(strategy="most_frequent", random_state=SEED))])
dummy.fit(X_train, y_train)
y_pred_dummy = dummy.predict(X_test)

print(f"[Dummy] Acurácia: {accuracy_score(y_test, y_pred_dummy):.3f}")
print(confusion_matrix(y_test, y_pred_dummy))
print(classification_report(y_test, y_pred_dummy))

# 5 métricas padrao do projeto
probs_dummy = dummy.predict_proba(X_test)[:, 1]
metrics_dummy = {
    "f1":        round(f1_score(y_test, y_pred_dummy, zero_division=0), 4),
    "roc_auc":   round(roc_auc_score(y_test, probs_dummy), 4),
    "pr_auc":    round(average_precision_score(y_test, probs_dummy), 4),
    "precision": round(precision_score(y_test, y_pred_dummy, zero_division=0), 4),
    "recall":    round(recall_score(y_test, y_pred_dummy, zero_division=0), 4),
}
print("[Dummy] Métricas:", metrics_dummy)

[Dummy] Acurácia: 0.735
[[1035    0]
 [ 374    0]]
              precision    recall  f1-score   support

           0       0.73      1.00      0.85      1035
           1       0.00      0.00      0.00       374

    accuracy                           0.73      1409
   macro avg       0.37      0.50      0.42      1409
weighted avg       0.54      0.73      0.62      1409

[Dummy] Métricas: {'f1': 0.0, 'roc_auc': 0.5, 'pr_auc': 0.2654, 'precision': 0.0, 'recall': 0.0}


In [6]:
# --- LogisticRegression: baseline linear — referência mínima para o MLP superar ---
# class_weight='balanced' compensa o desbalanceamento (~26% churn) ajustando os pesos das classes
lr = Pipeline([("pre", preprocessor), ("clf", LogisticRegression(random_state=SEED))])
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

print(f"[LogisticRegression] Acurácia: {accuracy_score(y_test, y_pred_lr):.3f}")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))

# 5 métricas padrao do projeto
probs_lr = lr.predict_proba(X_test)[:, 1]
metrics_lr = {
    "f1":        round(f1_score(y_test, y_pred_lr, zero_division=0), 4),
    "roc_auc":   round(roc_auc_score(y_test, probs_lr), 4),
    "pr_auc":    round(average_precision_score(y_test, probs_lr), 4),
    "precision": round(precision_score(y_test, y_pred_lr, zero_division=0), 4),
    "recall":    round(recall_score(y_test, y_pred_lr, zero_division=0), 4),
}
print("[LR] Métricas:", metrics_lr)

[LogisticRegression] Acurácia: 0.802
[[917 118]
 [161 213]]
              precision    recall  f1-score   support

           0       0.85      0.89      0.87      1035
           1       0.64      0.57      0.60       374

    accuracy                           0.80      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.80      0.80      1409

[LR] Métricas: {'f1': 0.6043, 'roc_auc': 0.8487, 'pr_auc': 0.645, 'precision': 0.6435, 'recall': 0.5695}


## 4. Validação cruzada + MLflow tracking

In [7]:
# StratifiedKFold mantém a proporção de churn em cada fold — essencial para datasets desbalanceados
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
SCORING = ["f1", "roc_auc", "average_precision", "precision", "recall"]  # 5 métricas padrão do projeto

mlflow.set_tracking_uri("../mlruns")
mlflow.set_experiment("churn-baselines")

baselines = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent", random_state=SEED),
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED),
}

results = {}
for name, clf in baselines.items():
    pipe = Pipeline([("pre", preprocessor), ("clf", clf)])
    scores = cross_validate(pipe, X, y, cv=cv, scoring=SCORING, n_jobs=1)
    metrics = {k: scores[f"test_{k}"].mean() for k in SCORING}
    # renomeia average_precision -> pr_auc para consistência com o restante do projeto
    metrics["pr_auc"] = metrics.pop("average_precision")

    with mlflow.start_run(run_name=name):
        mlflow.log_params({"model": name, "cv_folds": 5, "seed": SEED})
        mlflow.log_metrics(metrics)

    results[name] = metrics
    logger.info("[%s] %s", name, metrics)

INFO:__main__:[dummy_most_frequent] {'f1': np.float64(0.0), 'roc_auc': np.float64(0.5), 'precision': np.float64(0.0), 'recall': np.float64(0.0), 'pr_auc': np.float64(0.2653698424091877)}
INFO:__main__:[logistic_regression] {'f1': np.float64(0.6414354549477803), 'roc_auc': np.float64(0.8575787143559934), 'precision': np.float64(0.529836521484818), 'recall': np.float64(0.8127439033132141), 'pr_auc': np.float64(0.673338732602754)}


In [8]:
pd.DataFrame(results).T.round(4)

,f1,roc_auc,precision,recall,pr_auc
dummy_most_frequent,0.0000,0.5000,0.0000,0.0000,0.2654
logistic_regression,0.6414,0.8576,0.5298,0.8127,0.6733
